# Data Understanding: Brazilian E-Commerce Dataset

**Objective**

This notebook explores the structure and relationships of the Olist Brazilian E-Commerce dataset.

The main objectives are:

* understand the granularity of each table;
* identify important identifiers and relationships;
* investigate potential data quality issues;
* document assumptions and questions for downstream analysis.

Rather than performing business analysis at this stage, this notebook focuses on understanding how the raw tables should be interpreted and prepared for further analysis.

### 0. Schema
<img src="../figures/schema.png" width="70%">  

### 1. DataFrames Initialization

In [2]:
import pandas as pd
import sys
from pathlib import Path

# Add the project root to Python's import path so modules in src/ can be imported.
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
orders = pd.read_csv(r'..\data\raw\olist_orders_dataset.csv', index_col='order_id', 
                        parse_dates=[3, 4, 5, 6, 7], date_format="%Y-%m-%d %H:%M:%S")
items = pd.read_csv(r'..\data\raw\olist_order_items_dataset.csv', index_col='order_id', 
                    parse_dates=['shipping_limit_date'], date_format="%Y-%m-%d %H:%M:%S")
sellers = pd.read_csv(r'..\data\raw\olist_sellers_dataset.csv', index_col='seller_id')
geolocations = pd.read_csv(r'..\data\raw\olist_geolocation_dataset.csv', index_col='geolocation_zip_code_prefix')
customers = pd.read_csv(r'..\data\raw\olist_customers_dataset.csv', index_col='customer_id')
reviews = pd.read_csv(r'..\data\raw\olist_order_reviews_dataset.csv', index_col='order_id', 
                        parse_dates=['review_creation_date', 'review_answer_timestamp'], 
                        date_format="%Y-%m-%d %H:%M:%S")
payments = pd.read_csv(r'..\data\raw\olist_order_payments_dataset.csv', index_col='order_id')
products = pd.read_csv(r'..\data\raw\olist_products_dataset.csv', index_col='product_id')
cat_trans = pd.read_csv(r'..\data\raw\product_category_name_translation.csv', 
                        index_col='product_category_name') 

### 2. orders 

Granularity:  
One row represents one order.

Index:  
order_id

In [4]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99441 entries, e481f51cbdc54678b7cc49136f2d6af7 to 66dea50a8b16d9b4dee7af250b4be1a5
Data columns (total 7 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   customer_id                    99441 non-null  object        
 1   order_status                   99441 non-null  object        
 2   order_purchase_timestamp       99441 non-null  datetime64[ns]
 3   order_approved_at              99281 non-null  datetime64[ns]
 4   order_delivered_carrier_date   97658 non-null  datetime64[ns]
 5   order_delivered_customer_date  96476 non-null  datetime64[ns]
 6   order_estimated_delivery_date  99441 non-null  datetime64[ns]
dtypes: datetime64[ns](5), object(2)
memory usage: 6.1+ MB


In [5]:
orders.index.duplicated().sum()

np.int64(0)

In [6]:
orders.groupby("order_status")[
    [
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date"
    ]
].apply(lambda df: df.notna().mean())

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,1.000000,0.000000,0.000000
canceled,0.774400,0.120000,0.009600
created,0.000000,0.000000,0.000000
delivered,0.999855,0.999979,0.999917
invoiced,1.000000,0.000000,0.000000
processing,1.000000,0.000000,0.000000
shipped,1.000000,1.000000,0.000000
unavailable,1.000000,0.000000,0.000000


In [7]:
orders['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

**Observation**

- `order_status` records the status associated with each order. Most orders are marked as `delivered`, while a smaller proportion remain in intermediate states (e.g., `processing`, `shipped`, `invoiced`) or terminated states (e.g., `canceled`, `unavailable`).

- Four timestamp columns (`order_purchase_timestamp`, `order_approved_at`, `order_delivered_carrier_date`, and `order_delivered_customer_date`) describe different stages of the order fulfillment process.

- The number of non-null values decreases as the order progresses through the fulfillment process.

- `order_purchase_timestamp` and `order_estimated_delivery_date` contain the same number of non-null values.

- Both estimated and actual delivery dates are available, making it possible to evaluate delivery performance for eligible orders.

**Questions**

- How are different `order_status` values related to the availability of the timestamp columns?

- Which order statuses should be included in different analyses (e.g., delivery performance or customer satisfaction)?

- Why are timestamps missing in later stages of the order lifecycle? Are they fully explained by the order status?

- Does delivery performance influence customer review scores?

- Can a single order involve multiple products, therefore multiple sellers, and if so, how is the delivery date recorded in the orders table?


**Working Hypotheses**

- Missing timestamps in later stages of the order lifecycle are expected for orders that have not progressed to those stages. The relationship between `order_status` and missing timestamps should be verified.

- Delivery-related analyses should initially focus on orders with `order_status == 'delivered'`, while the treatment of orders in other statuses should be determined according to the analytical objective.

- `order_estimated_delivery_date` is likely generated after an order is placed rather than after it is delivered.

- The delivery timestamp stored in the orders table may correspond to the completion of the entire order rather than individual items, which could complicate analyses when an order contains products from multiple sellers.

### 3. items

Granularity:  
One row represents one item in an order.

Index:  
order_id

In [8]:
items.info()

<class 'pandas.core.frame.DataFrame'>
Index: 112650 entries, 00010242fe8c5a6d1ba2dd792cb16214 to fffe41c64501cc87c801fd61db3f6244
Data columns (total 6 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_item_id        112650 non-null  int64         
 1   product_id           112650 non-null  object        
 2   seller_id            112650 non-null  object        
 3   shipping_limit_date  112650 non-null  datetime64[ns]
 4   price                112650 non-null  float64       
 5   freight_value        112650 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(2)
memory usage: 6.0+ MB


In [9]:
items.index.duplicated().sum()

np.int64(13984)

**Observation** 
- The table contains 112,650 order items, which is greater than the number of orders because each row represents a single item within an order.
- The DataFrame index (`order_id`) is therefore non-unique, reflecting a one-to-many relationship between orders and order items. Selecting by `order_id` may return multiple rows.
- Each row represents one purchased item rather than an entire order. The table does not contain a quantity column; each row represents an item within an order.

**Initial Questions**

- How many items does a typical order contain?
- Are shipping costs proportional to the number of items or their total price?

### 4. customers

Granularity:  
Granularity: One row represents the customer information recorded for an order.

Index:  
customer_id

In [10]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99441 entries, 06b8999e2fba1a1fbc88172c00ba8bc7 to 274fa6071e5e17fe303b9748641082c8
Data columns (total 4 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   customer_unique_id        99441 non-null  object
 1   customer_zip_code_prefix  99441 non-null  int64 
 2   customer_city             99441 non-null  object
 3   customer_state            99441 non-null  object
dtypes: int64(1), object(3)
memory usage: 3.8+ MB


In [11]:
pd.Series({
    "Duplicated customers.index": customers.index.duplicated().sum(),
    "Duplicated customer_unique_id": customers["customer_unique_id"].duplicated().sum()
})

Duplicated customers.index          0
Duplicated customer_unique_id    3345
dtype: int64

Question: Is there a one-to-one relationship between `orders.customer_id` and `customers.customer_id`?

In [12]:
[orders.index.is_unique, customers.index.is_unique, set(orders["customer_id"]) == set(customers.index)]


[True, True, True]

**Observations**
- Two customer identifiers are provided: `customer_id` (the DataFrame index) and `customer_unique_id`. The index (`customer_id`) is unique, while `customer_unique_id` is not. The number of customer records matches the number of orders, and `customer_id` is also unique in the `orders` DataFrame.
- Each customer record is associated with a ZIP code prefix, city, and state, providing a recorded geographic location for each customer record.

**Initial Questions**

- What is the distinction between `customer_id` and `customer_unique_id`? Which identifier should be used in customer-level analyses?
- Does the recorded customer location represent the customer's registered address or the delivery location of an order? If they differ, where is the delivery location recorded?

**Working Hypotheses**

- `customer_id` may identify a customer record associated with an individual order, whereas `customer_unique_id` may identify the underlying customer across multiple purchases. This hypothesis is motivated by the equal number of customer_id records in the `customers` and `orders` DataFrames, the uniqueness of `customer_id` in both tables, and the non-uniqueness of `customer_unique_id`.
- The recorded customer location may represent a default or registered address rather than the delivery destination of every order. If customers are able to ship orders to different addresses, additional delivery location information may exist elsewhere in the dataset or may not be available.

### 5. sellers
Granularity:  
One row represents one seller.

Index:  
seller_id

In [13]:
sellers.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3095 entries, 3442f8959a84dea7ee197c632cb2df15 to 9e25199f6ef7e7c347120ff175652c3b
Data columns (total 3 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   seller_zip_code_prefix  3095 non-null   int64 
 1   seller_city             3095 non-null   object
 2   seller_state            3095 non-null   object
dtypes: int64(1), object(2)
memory usage: 96.7+ KB


**Observations**

- The table contains 3,095 sellers.
- Each seller is uniquely identified by `seller_id`, which is used as the DataFrame index.
- Only geographic information (ZIP code prefix, city, and state) is provided. No seller-level business metrics, such as registration date or seller category, are available.

**Initial Questions**

- How are sellers geographically distributed across Brazil?
- How many orders are associated with each seller?
- Does seller location relate to delivery performance or customer satisfaction after merging with other tables?

### 6. geolocations

Granularity:  
One row represents one geolocation record, rather than one unique ZIP code prefix or one unique location.

Initial index:  
zip_code_prefix (uniqueness assumption breaks after inspection)

Working index:  
default RangeIndex

In [14]:
geolocations.columns = pd.Index(['lat', 'lng', 'city', 'state'])
geolocations.index.name = 'zip_code_prefix'
geolocations.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1000163 entries, 1037 to 99950
Data columns (total 4 columns):
 #   Column  Non-Null Count    Dtype  
---  ------  --------------    -----  
 0   lat     1000163 non-null  float64
 1   lng     1000163 non-null  float64
 2   city    1000163 non-null  object 
 3   state   1000163 non-null  object 
dtypes: float64(2), object(2)
memory usage: 38.2+ MB


Question: Is `zip_code_prefix` duplicated?

>Background
>
>Brazilian postal codes (CEP) contain eight digits, while the Olist dataset provides only the first five digits as `zip_code_prefix`. The ZIP code prefix therefore represents a broader geographic area rather than a complete postal-code location and may correspond to multiple geographic records or more than one city. Therefore, `zip_code_prefix` should not automatically be treated as a unique geographic identifier.

In [15]:
geolocations.index.duplicated().sum()

np.int64(981148)

`zip_code_prefix` is severely duplicated and is therefore not suitable as the DataFrame index. It is restored as a regular column, and the default `RangeIndex` is used for subsequent analysis.

In [16]:
geolocations.reset_index(inplace=True)

Question: What varies among records sharing the same ZIP prefix?

Records sharing the same prefix may differ in city, state, or geographic coordinates. The following check therefore counts the number of distinct cities, states, and coordinate pairs associated with each ZIP code prefix.

In [17]:
geo_check = (geolocations.groupby("zip_code_prefix")
            .agg(n_cities=("city", "nunique"), n_states=("state", "nunique"),)
)

n_unique_coordinates = (
    geolocations[["zip_code_prefix", "lat", "lng"]]
    .drop_duplicates()
    .groupby("zip_code_prefix")
    .size()
    .rename("n_unique_coordinates")
)

geo_check = geo_check.join(n_unique_coordinates)

geo_check.sort_values(["n_cities", "n_states", "n_unique_coordinates"],ascending=False).head(10)

,n_cities,n_states,n_unique_coordinates
zip_code_prefix,,,
6900,5,1,380
28950,5,1,274
42850,5,1,178
13454,5,1,118
13455,5,1,71
13457,5,1,42
13318,5,1,41
78290,5,1,26
17970,5,1,14


Question: Since customer and seller records provide city and state in addition to ZIP prefix, can all three fields be used together to identify a more specific geographic area?

The following counts how many unique coordinate pairs remain for each distinct `(zip_code_prefix, state, city)` combination.

In [18]:
triplet_check = (geolocations[ [ "zip_code_prefix", "state", "city", "lat", "lng" ] ] 
                    .drop_duplicates() 
                    .groupby( ["zip_code_prefix", "state", "city"] ) 
                    .size() 
                    .rename("n_unique_coordinates") 
                    .reset_index() 
)

triplet_check.head(5)

,zip_code_prefix,state,city,n_unique_coordinates
0,1001,SP,sao paulo,8
1,1001,SP,são paulo,3
2,1002,SP,sao paulo,5
3,1002,SP,são paulo,1
4,1003,SP,sao paulo,10


Question: Are the three components represented consistently enough for exact matching? In particular, given that `city` is free-form text, does it contain inconsistent representations?

The usefulness of the triplet also depends on whether its components are represented consistently. `state` uses standardized abbreviations, whereas `city` is free-form text and may contain spelling or formatting differences. To inspect this issue, `city` names are compared within each `(zip_code_prefix, state`) group. 

In [19]:
city_check = (geolocations.groupby(["zip_code_prefix", "state"]) 
                .agg( n_cities=("city", "nunique"), 
                        cities=("city", lambda x: sorted(x.dropna().unique())) ) )
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
city_check[city_check["n_cities"] > 1 ].sort_values("n_cities", ascending=False).head(15)

,,n_cities,cities
zip_code_prefix,state,,
13454,SP,5,"[santa barbara d oeste, santa barbara d'oeste, santa bárbara d'oeste, santa bárbara d`oeste, santa bárbara doeste]"
13318,SP,5,"[cabreuva, jacare, jacare (cabreuva), jacaré, jacaré (cabreúva)]"
13455,SP,5,"[santa barbara d oeste, santa barbara d'oeste, santa barbara doeste, santa bárbara d'oeste, santa bárbara doeste]"
17970,SP,5,"[sao joao do pau d alho, sao joao do pau d'alho, sao joao do pau dalho, são joão do pau d%26apos%3balho, são joão do pau d'alho]"
6900,SP,5,"[embu guacu, embu guaçu, embu-guacu, embu-guaçu, embuguacu]"
28950,RJ,5,"[armacao de buzios, armacao dos buzios, armação dos búzios, buzios, búzios]"
13457,SP,5,"[santa barbara d oeste, santa barbara d'oeste, santa barbara doeste, santa bárbara d'oeste, santa bárbara doeste]"
42850,BA,5,"[dias d avila, dias d'avila, dias d'ávila, dias davila, dias dávila]"
78290,MT,5,"[figueiropolis d oeste, figueiropolis d'oeste, figueiropolis doeste, figueirópolis d'oeste, figueirópolis doeste]"


Question: Is the multiplicity within each `(zip_code_prefix, state)` group caused by inconsistent textual representations?

City names are normalized before distinct strings are interpreted as distinct geographic locations.

In [20]:
from src.geo_clean import normalize_city

geolocations["city_norm"] = (
    geolocations["city"]
    .map(normalize_city)
)

After normalization, the number of distinct city names within each `(zip_code_prefix, state)` group is calculated again. The raw city values are retained alongside the normalized values so that any collapsed or remaining differences can be inspected.

In [21]:
city_norm_check = (
    geolocations
    .groupby(["zip_code_prefix", "state"])
    .agg(
        n_cities=("city_norm", "nunique"),

        cities_norm=(
            "city_norm",
            lambda x: sorted(x.dropna().unique())
        ),

        cities_raw=(
            "city",
            lambda x: sorted(x.dropna().unique())
        )
    )
)

city_norm_check = (
    city_norm_check[
        city_norm_check["n_cities"] > 1
    ]
    .sort_values("n_cities", ascending=False)
)

city_norm_check.head(5)

,,n_cities,cities_norm,cities_raw
zip_code_prefix,state,,,
45936,BA,4,"[itabata, itabatan, itabatan (mucuri), mucuri]","[itabata, itabatan, itabatan (mucuri), mucuri]"
65935,MA,4,"[buritirama, buritirana, senador la rocque, senador la roque]","[buritirama, buritirana, senador la rocque, senador la roque]"
38749,MG,3,"[patrocinio, sao joao da serra negra, silvano]","[patrocinio, sao joao da serra negra, silvano]"
36206,MG,3,"[costas da mantiqueira, mantiqueira do palmital, sao sebastiao dos torres]","[costas da mantiqueira, mantiqueira do palmital, são sebastião dos torres]"
28950,RJ,3,"[armacao de buzios, armacao dos buzios, buzios]","[armacao de buzios, armacao dos buzios, armação dos búzios, buzios, búzios]"


**Observations**

- Records sharing the same `zip_code_prefix` may differ in city, state, and geographic coordinates. The relationship between ZIP code prefixes and geographic information is therefore not one-to-one.

- The triplet `(zip_code_prefix, state, city)`, which is also available in the customer and seller tables, provides a more specific basis for geographic matching. However, the combination may still correspond to multiple coordinate pairs.

- The `city` field is free-form text and contains inconsistent representations. Some multiplicity within `(zip_code_prefix, state)` groups arise from differences in spelling or formatting rather than genuinely different municipalities.

- Normalizing `city` names reduces some textual inconsistencies, but multiple normalized city names still remain within some `(zip_code_prefix, state)` groups. These remaining cases cannot be resolved reliably from the Olist geolocation table alone.

**Initial Questions**

- After normalizing city names, which remaining multiple-city mappings represent genuine municipality differences and which reflect errors or inconsistencies in the geolocation data?

- How consistent are the Olist ZIP-prefix, city, and state combinations with official Brazilian geographic or postal-code sources?

- When a `(zip_code_prefix, state, city)` combination corresponds to multiple geographic coordinates, how should a representative location be assigned for subsequent analysis?

### 7. reviews

Granularity:  
One row represents one review record associated with an order.

Index:  
order_id

In [22]:
reviews.info()

<class 'pandas.core.frame.DataFrame'>
Index: 99224 entries, 73fc7af87114b39712e6da79b0a377eb to 90531360ecb1eec2a1fbb265a0db0508
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   review_id                99224 non-null  object        
 1   review_score             99224 non-null  int64         
 2   review_comment_title     11568 non-null  object        
 3   review_comment_message   40977 non-null  object        
 4   review_creation_date     99224 non-null  datetime64[ns]
 5   review_answer_timestamp  99224 non-null  datetime64[ns]
dtypes: datetime64[ns](2), int64(1), object(3)
memory usage: 5.3+ MB


Question: Can duplicated order IDs in the reviews dataframe be explained by orders containing multiple items?

In [23]:
reviews.index.duplicated().sum()

np.int64(551)

In [24]:
pd.Series({
'Duplicated order_ID in this dataframe': reviews.index.duplicated().sum(),
'Duplicated order_ID in the items dataframe': items.index.duplicated().sum()
})

Duplicated order_ID in this dataframe           551
Duplicated order_ID in the items dataframe    13984
dtype: int64

Question: How do review records differ when they share the same order ID?

In [25]:
reviews.loc[reviews.index.duplicated(keep=False)].sort_index()

,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
order_id,,,,,,
0035246a40f520710769010f752e7507,89a02c45c340aeeb1354a24e7d4b2c1e,5,NaN,NaN,2017-08-29 00:00:00,2017-08-30 01:59:12
0035246a40f520710769010f752e7507,2a74b0559eb58fc1ff842ecc999594cb,5,NaN,"Estou acostumada a comprar produtos pelo baratheon. Super confiável, transparência, qualidade e agilidade. Super recomendo.",2017-08-25 00:00:00,2017-08-29 21:45:57
013056cfe49763c6f66bda03396c5ee3,ab30810c29da5da8045216f0f62652a2,5,NaN,NaN,2018-02-22 00:00:00,2018-02-23 12:12:30
013056cfe49763c6f66bda03396c5ee3,73413b847f63e02bc752b364f6d05ee9,4,NaN,NaN,2018-03-04 00:00:00,2018-03-05 17:02:00
0176a6846bcb3b0d3aa3116a9a768597,830636803620cdf8b6ffaf1b2f6e92b2,5,NaN,NaN,2017-12-30 00:00:00,2018-01-02 10:54:06
0176a6846bcb3b0d3aa3116a9a768597,d8e8c42271c8fb67b9dad95d98c8ff80,5,NaN,NaN,2017-12-30 00:00:00,2018-01-02 10:54:47
02355020fd0a40a0d56df9f6ff060413,017f0e1ea6386de662cbeba299c59ad1,1,NaN,ja reclamei varias vezes e ate hoje não sei onde foi parar o produto que comprei. quero cancelar esta compra e quero meu dinheiro de volta.,2018-03-29 00:00:00,2018-03-30 03:16:19
02355020fd0a40a0d56df9f6ff060413,0c8e7347f1cdd2aede37371543e3d163,3,NaN,UM DOS PRODUTOS (ENTREGA02) COMPRADOS NESTE PEDIDO NAO RECEBI AINDA E OUTRO (ENTREGA01) RECEBI EMAIL QUE ESTA COM DIFICULDADES NA ENTREGA E NÃO TIVE MAIS NOTICIAS,2018-03-21 00:00:00,2018-03-22 01:32:08
029863af4b968de1e5d6a82782e662f5,61fe4e7d1ae801bbe169eb67b86c6eda,4,NaN,NaN,2017-07-19 00:00:00,2017-07-20 12:06:11


Question: Is there evidence that some review records are automatically generated after a fixed period?

In [26]:
time_lapse = reviews['review_answer_timestamp'] - reviews['review_creation_date']
time_lapse.value_counts().sort_values(ascending=False)

1 days 00:44:33      7
0 days 23:05:28      6
1 days 00:36:16      6
1 days 00:43:35      6
1 days 10:53:21      6
0 days 23:36:35      6
0 days 23:14:11      6
1 days 01:06:17      6
1 days 11:07:08      6
0 days 23:22:05      6
1 days 01:17:50      6
0 days 23:08:09      6
1 days 01:47:15      5
0 days 23:14:34      5
0 days 23:11:57      5
3 days 11:22:24      5
0 days 23:34:11      5
1 days 00:34:22      5
0 days 21:07:41      5
1 days 01:02:05      5
0 days 23:31:23      5
1 days 02:00:55      5
1 days 00:34:49      5
1 days 11:16:13      5
1 days 01:35:25      5
0 days 18:16:18      5
1 days 00:09:47      5
1 days 01:05:00      5
2 days 22:26:00      5
1 days 09:41:03      5
1 days 12:44:51      5
0 days 23:25:02      5
0 days 23:10:43      5
1 days 11:03:52      5
0 days 21:38:48      5
1 days 00:07:46      5
3 days 11:26:40      5
1 days 00:54:52      5
1 days 11:54:55      5
0 days 23:36:13      5
1 days 01:03:41      5
0 days 23:26:49      5
1 days 00:42:11      5
0 days 22:4

**Observations**

- Only a minority of reviews contain a comment message, and an even smaller subset contains a comment title.
- A preliminary inspection suggests that most comment texts are written in Portuguese, so translation or multilingual text processing may be required for later textual analysis.
- The number of duplicated order IDs in reviews differs from that in `items`, suggesting that multiple products per order cannot fully explain duplicated review records.
- `review_score` is an integer rating from 1 to 5, where 5 represents the highest level of satisfaction.
- Some orders are associated with multiple reviews, which are generally created or answered at different timestamps.

**Initial Questions**

- How can review comments be translated and analyzed to extract sentiment, topics, and specific service issues?
- Do orders without customer feedback receive a default review score, or are they excluded from the review table?
- Why do some orders have multiple review records? Do they correspond to different items, multiple review submissions, or later updates to an earlier review?

**Working Hypotheses**

- Multiple review records associated with the same order may be related to multiple items within the order, although the review table itself contains no item-level identifier to verify this directly.

- The distribution of `time_lapse` shows a large number of unique intervals. The most frequent interval occurring only seven times, while over 82,000 different intervals exist. This does not provide evidence of a systematic automatic generation process.

- Since the number of duplicated `order_id` values differs between the `reviews` and `items` DataFrames, the relationship is unlikely to be explained solely by the number of purchased items. Further investigation is required.

### 8. payments


Granularity:  
One row represents one payment record.

Index:  
order_id

In [27]:
payments.info()

<class 'pandas.core.frame.DataFrame'>
Index: 103886 entries, b81ef226f3fe1789b1e8b2acac839d17 to 28bbae6599b09d39ca406b747b6632b1
Data columns (total 4 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   payment_sequential    103886 non-null  int64  
 1   payment_type          103886 non-null  object 
 2   payment_installments  103886 non-null  int64  
 3   payment_value         103886 non-null  float64
dtypes: float64(1), int64(2), object(1)
memory usage: 4.0+ MB


In [28]:
payments.index.duplicated().sum()

np.int64(4446)

Question: How do payment records differ when they share the same order ID?

In [29]:
payments[payments.index.duplicated(keep=False)].sort_index()

,payment_sequential,payment_type,payment_installments,payment_value
order_id,,,,
0016dfedd97fc2950e388d2971d718c7,2,voucher,1,17.92
0016dfedd97fc2950e388d2971d718c7,1,credit_card,5,52.63
002f19a65a2ddd70a090297872e6d64e,1,voucher,1,44.11
002f19a65a2ddd70a090297872e6d64e,2,voucher,1,33.18
0071ee2429bc1efdc43aa3e073a5290e,2,voucher,1,92.44
0071ee2429bc1efdc43aa3e073a5290e,1,voucher,1,100.00
009ac365164f8e06f59d18a08045f6c4,2,voucher,1,4.50
009ac365164f8e06f59d18a08045f6c4,6,voucher,1,4.17
009ac365164f8e06f59d18a08045f6c4,4,voucher,1,5.45


Question: Can duplicated order IDs in the payment dataframe be explained by payment sequential is larger than 1?

In [30]:
payments[payments['payment_sequential'] > 1].index

Index(['5cfd514482e22bc992e7693f0e3e8df7', '3689194c14ad4e2e7361ebd1df0e77b0',
       '21b8b46679ea6482cbf911d960490048', 'ea9184ad433a404df1d72fa0a8764232',
       '82ffe097d8ddbf319a523b9bbe7725d5', '487c1451b8fd7347d0e80e5aca887e91',
       '65c863d90026e9dd4888e28d2135c983', '591083bc42b589c7052118aa83118e76',
       '79b63d1f9cf0257b33b04fbfb0847cba', '332c6742772f2df936696b6512b10edb',
       ...
       '08fa8b49ff198d1332df4668087150ed', '71853944ebfe6bd5f5de0302cba14354',
       '67937bb4e0c9a4156fe47c89382d33e7', 'e2da042c42ce790c81ef3a9f666a92b6',
       '1ea11b975aea60ca92fa0559184d3d34', 'fedcd9f7ccdc8cba3a18defedd1a5547',
       'f0a5b7c94819c69d12a1c0458ec74756', 'cf101c3abd3c061ca9f78c1bbb1125af',
       'd0bff47153ef056bb4f884a2ec2f0691', '31bc09fdbd701a7a4f9b55b5955b8687'],
      dtype='object', name='order_id', length=4526)

In [31]:
multiple_payment_sequences = payments[payments['payment_sequential'] > 1].sort_index()
multiple_payment_sequences

,payment_sequential,payment_type,payment_installments,payment_value
order_id,,,,
0016dfedd97fc2950e388d2971d718c7,2,voucher,1,17.92
002f19a65a2ddd70a090297872e6d64e,2,voucher,1,33.18
0071ee2429bc1efdc43aa3e073a5290e,2,voucher,1,92.44
009ac365164f8e06f59d18a08045f6c4,2,voucher,1,4.50
009ac365164f8e06f59d18a08045f6c4,6,voucher,1,4.17
009ac365164f8e06f59d18a08045f6c4,4,voucher,1,5.45
009ac365164f8e06f59d18a08045f6c4,5,voucher,1,8.75
009ac365164f8e06f59d18a08045f6c4,3,voucher,1,8.25
00ac05fe0fc047c54418098eb64e3aaa,2,debit_card,1,123.47


Question: Is  payment sequential is larger than 1 always associated with different payment types?

In [32]:
multiple_payment_sequences.groupby('order_id')['payment_type'].nunique().sort_values()

order_id
febdd4292b7bc064ae8e10f572aa2ce9    1
fb89dbec79bc8b1f0d12165058cb1cb3    1
fbacbe04cbdc1c9aadf3409a9aa1e9a0    1
fbbc0a864330509c41ca4b4f0ff9b780    1
fbc32bc4a800352a5c9d8a57828b4727    1
fbd05af8c1751b972fc6e19b1190990e    1
fbe3a31675cc6b48eff1c765ae6bfb06    1
fbf352b18a26c6a86a53cca922facd25    1
fbf7da560930fa17715f1fa8e0b4ac81    1
fbfb14bfcbaabcc1473524656c76a4cb    1
fbff77db5421bb7e1da292db45d07e75    1
fc297415452e5dc0d06afcd16e864ca8    1
fc2a4d7b806b8d5572a052da4a5a6bed    1
fc2f86d56413e4c2e64ffd50117f6a6b    1
fc62afb57b86ee5363aee2b7c04421c0    1
fc6e9a6cd6985d3d454cccfdfdf66931    1
fc6f84b0e39444f7c25d54107317faca    1
fc965b1e986dea6cd20ac94c50878a75    1
fcab9e8b311ef84e1dce97a58af5b7c5    1
fccf0d3397491d555ae0d9a4689bfc29    1
fce3db7092e1f432d0ec790414b6a5e4    1
fcf9b8fad0bf8c9ebbe6ac43a434d0d9    1
fcfc50a9520809adee86b167b0cf297e    1
fd10deeb9842f5c285152824f810fc08    1
fedcd9f7ccdc8cba3a18defedd1a5547    1
fd2b75f431649ee3de573b444d320f48    1
fd5

**Observations**
-  The index of the `payments` dataframe (`order_id`) has duplicates. 
- Some payment records have `payment_sequential > 1`. Since the sequence starts from 1, these orders are associated with multiple payment records. In addition, `payment_installments` can exceed 1, meaning that a payment may be split into multiple installments.
- Initial inspection suggests that credit cards and vouchers are common payment methods.

**Initial Questions**

- Under what circumstances does one order generate multiple payment records?
- Are duplicated payment records associated with multiple payment methods, multiple items within an order, or other payment processes?


**Working Hypotheses**

- Initial inspection shows that orders with multiple payment types are associated with `payment_sequential > 1`, suggesting that this is one mechanism that produces multiple payment records for a single order.

- However, duplicated payment records are also observed when all records share the same `payment_type`. 
Additional factors may contribute to multiple payment records and will be investigated by combining the payment table with other related tables in the next notebook.

### 9. products

Granularity:  
One row represents one product.

Index:  
product_id

In [33]:
products['product_category_name'] = products['product_category_name'].map(
                    cat_trans["product_category_name_english"])

In [34]:
products.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32951 entries, 1e9e8ef04dbcff4541ed26657ea517e5 to 106392145fca363410d287a815be6de4
Data columns (total 8 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_category_name       32328 non-null  object 
 1   product_name_lenght         32341 non-null  float64
 2   product_description_lenght  32341 non-null  float64
 3   product_photos_qty          32341 non-null  float64
 4   product_weight_g            32949 non-null  float64
 5   product_length_cm           32949 non-null  float64
 6   product_height_cm           32949 non-null  float64
 7   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(1)
memory usage: 2.3+ MB


In [35]:
products.index.duplicated().sum()

np.int64(0)

In [36]:
# Save `products` for later use.

products.to_parquet(
    r"../data/processed/products.parquet",
    index=True,
)

**Observations**

- The DataFrame's index (`product_id`) uniquely identifies each product.
- Products are classified into predefined categories with translation provided by the dataset.
- The DataFrame contains both textual metadata (name length, description length, photo count) and physical characteristics (weight and dimensions).
- This table provides product-level attributes that can be merged into transaction data if product characteristics become relevant during later analyses.

**Initial Questions**

- Do certain product categories receive better reviews or experience longer delivery times?
- How strongly are product weight and dimensions associated with freight charges?
- Does the amount of product information (description length or number of photos) influence customer satisfaction?
- How should missing product categories be handled?

### Summary and Next Steps

#### Summary

The exploratory analysis revealed several characteristics of the dataset that should be considered during data preparation and subsequent analysis.

**Order lifecycle**

* The `orders` table contains orders with different `order_status` values, indicating that not all orders have completed the full fulfillment process. Missing timestamps in later stages of the order lifecycle may therefore reflect differences in order status rather than data quality issues. The relationship between `order_status` and timestamp availability should be verified before conducting delivery-related analyses.

* Each order has only one set of order-level timestamps, regardless of how many items it contains. Consequently, the interpretation of delivery-related metrics should account for the possibility that an order contains multiple items or sellers.


**Order-item relationship**

* The `items` table contains multiple records for some `order_id`s because a single order may contain multiple items. Aggregation rules will therefore be required when constructing order-level datasets.

**Geographic information**

* The geolocation table contains inconsistent representations in the `city` column, indicating a data-quality issue that should be corrected by referencing official Brazilian geographic or postal-code sources before downstream geographic analysis.

**Customer identification**

* `customer_id` is a unique identifier associated with a single order record.
* `customer_unique_id` identifies the underlying customer and may appear across multiple orders.

Consequently, customer-level analyses should be based on `customer_unique_id`, whereas order-level analyses can use `customer_id`.

**Review records**

* Some `order_id`s appear multiple times in the reviews table. This duplication cannot be explained solely by orders containing multiple items and requires further investigation.
* No evidence was found that reviews or ratings are automatically generated after a fixed period of time.

**Product information**

* Some products have missing category information, which should be considered when conducting product- or category-level analyses.

#### Next steps

The next notebook will focus on constructing analysis-ready datasets. The main tasks include:

* validating and interpreting one-to-many relationships between tables;
* defining appropriate aggregation rules for different analytical objectives;
* merging tables according to the intended unit of analysis (e.g., order, customer, seller, or product);
* handling incorrect or incomplete information where necessary.


The dataset contains multiple entities connected through one-to-many relationships. The next step is not to combine all tables into a single dataset, but to construct analytical datasets according to specific business questions. Before doing so, relationships between tables need to be validated and aggregation strategies need to be defined.